In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import TensorDataset, DataLoader
from torchviz import make_dot
from torchsummary import summary

import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import re

from sklearn import svm
from sklearn.model_selection import train_test_split

import glob
import os
import time
import sys
import wandb
import json
from tqdm.notebook import tqdm

sys.path.append("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Binary_classification/training/models")
from Binary_classification import Binary_classification

In [ ]:
class DataSet():
    def __init__(self, data, label):
        self.label = label
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index], self.label[index]

In [ ]:
weight_path = "/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Binary_classification/training/save_dir/binary_cheting_model_test.pth"
weight_para = torch.load(weight_path, map_location=torch.device('cpu'))

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') 
print(f"使用デバイス: {device}")

In [ ]:
model = Binary_classification(latent=100, input_depth=30, input_height=100, input_width=100)
model.load_state_dict(weight_para)
model.to(device)
model.eval()

In [ ]:
summary(model.to(device), (1, 30, 100, 100))

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') 

In [ ]:
fits_path = "/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/FUGIN/12CO/FGN_01700+0000_2x2_12CO_v1.00_cube.fits"
bubble_data = np.load("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Binary_classification/data/FUGIN/processed_data/FGN_01700+0000/bubble_data.npy")
removal_data = np.load("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Binary_classification/data/FUGIN/processed_data/FGN_01700+0000/non_bubble_data.npy")

sample_size = len(removal_data) // 10
random_indices = np.random.choice(len(removal_data), size=sample_size, replace=False)
removal_data = [removal_data[i] for i in random_indices]

In [ ]:
bubble_label = [1] * len(bubble_data)
removal_label = [0] * len(removal_data)

print(len(bubble_label))
print(len(removal_label))

In [ ]:
COLS = 9
rows = (len(bubble_data) + COLS - 1) // COLS
data_index = 0

for line in range(rows):
    fig, axes = plt.subplots(1, COLS, figsize=(2 * COLS, 3))
    axes_flat = axes.flatten()
    
    for k in range(COLS):
        if data_index < len(bubble_data):
            data_num = data_index
                        
            ax = axes_flat[k]
            data = bubble_data[data_num]
            
            ax.imshow(np.sum(data, axis=0), cmap="jet") 
            # ax.set_title(f"{combinations[k]}")
            ax.axis("off")
            
            data_index += 1
        else:
            axes_flat[k].axis("off")

    plt.tight_layout()
    # plt.savefig(f"No.{line}.png")
    plt.show()

In [ ]:
data = np.concatenate((bubble_data, removal_data))
label = np.concatenate((bubble_label, removal_label))

In [ ]:
len(data)

In [ ]:
# label

In [ ]:
# data = torch.from_numpy(data).float()
# train_data, val_data, train_labels, val_labels = train_test_split(
#     data, label, test_size=0.2, random_state=42, stratify=label
# )
# val_data, test_data, val_labels, test_labels = train_test_split(
#     val_data, val_labels, test_size=0.25, random_state=42, stratify=val_labels)

# train_dataset    = DataSet(train_data, train_labels)
# train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# val_dataset      = DataSet(val_data, val_labels)
# val_dataloader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
# dataloader_dic   = {"train": train_dataloader, "val": val_dataloader}

test_dataset = DataSet(data, label)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
model.to(device)

# --- 変数の初期化 ---
test_correct_preds = 0
test_total_samples = 0
test_true_positives = 0
test_actual_positives = 0
test_predicted_positives = 0  # 【追加】モデルがPositive(1)と予測した総数

predicted_list = []

# テストループ
for images, labels in tqdm(test_dataloader):
    # バッチサイズを維持したままチャンネル数を1に設定
    images = images.view(-1, 1, 30, 100, 100)
    labels = labels.to(device).float()
    
    # 【注】推論時は通常 torch.no_grad() を推奨しますが、元のコードに合わせています
    with torch.set_grad_enabled(True):
        # モデルの出力を計算する
        images = images.float()
        output, latent = model(images.clone().to(device))
        output = output.squeeze()
        
        # 閾値0.5で0/1に変換
        predicted = (output > 0.5).float()
        # print(predicted) # デバッグ用出力（必要に応じてコメントアウト）

        predicted_list.extend(predicted.clone().detach().cpu().numpy().tolist())

        print(predicted)
        
        if False in (predicted == labels):
            print("!!")
        
        # 1. Accuracy用
        test_correct_preds += (predicted == labels).sum().item()
        test_total_samples += labels.size(0)
                    
        # 2. Recall用 (分子: TP, 分母: 実際の正例数)
        test_true_positives += ((predicted == 1) & (labels == 1)).sum().item()
        test_actual_positives += (labels == 1).sum().item()

        # 3. Precision用 【追加】 (分母: 予測した正例数)
        test_predicted_positives += (predicted == 1).sum().item()

# --- スコア計算 ---
# Accuracy
test_accuracy = test_correct_preds / test_total_samples if test_total_samples > 0 else 0.0

# Recall = TP / (TP + FN) = TP / Actual Positives
test_recall = test_true_positives / test_actual_positives if test_actual_positives > 0 else 0.0

# Precision = TP / (TP + FP) = TP / Predicted Positives 【追加】
test_precision = test_true_positives / test_predicted_positives if test_predicted_positives > 0 else 0.0

# 結果の表示
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"Precision: {test_precision:.4f}")

In [ ]:
print("Test Accuracy: {:.4f}, Test Recall: {:.4f}, Test Precision: {:.4f}".format(test_accuracy, test_recall, test_precision))

In [ ]:
print(f"test actual positives: {test_actual_positives}")
print(f"test true positives: {test_true_positives}")
print(f"test total samples: {test_total_samples}")
print(f"test correct preds: {test_correct_preds}")

# バブルデータの正誤判定確認

In [ ]:
judgement_list = []
for i in predicted_list:
    jud = int(i)
    if jud == 0:
        jud = "No"
    elif jud == 1:
        jud = "Bubble"
    judgement_list.append(jud)

In [ ]:
COLS = 9
rows = (len(bubble_data) + COLS - 1) // COLS
data_index = 0

for line in range(rows):
    fig, axes = plt.subplots(1, COLS, figsize=(2 * COLS, 2))
    axes_flat = axes.flatten()
    
    for k in range(COLS):
        if data_index < len(bubble_data):
            data_num = data_index
                        
            ax = axes_flat[k]
            data = bubble_data[data_num]
            
            ax.imshow(np.sum(data, axis=0), cmap="viridis") 
            ax.set_title(f"{judgement_list[data_num]}")
            ax.axis("off")
            
            data_index += 1
        else:
            axes_flat[k].axis("off")

    plt.tight_layout()
    plt.savefig(f"No.{line}.png")
    plt.show()

In [ ]:
axes_num = bubble_data[0].shape
print(axes_num)

In [ ]:
# fitsのPathからfitsの名前のみを取り出す
fits_name = fits_path.split("/")[-1]
region_name = fits_name.split("_")
region_name = region_name[0] + "_" + region_name[1]
print(region_name)

In [ ]:
axes_num = bubble_data[0].shape[0]
print(axes_num)

for index in range(len(bubble_data)):
    data = bubble_data[index]
    
    fig, axes = plt.subplots(1, axes_num, figsize=(axes_num*3, 1*4))
    for j, data_segment in enumerate(data):
        axes[j].imshow(data_segment, vmin=0, vmax=1, cmap="jet")
        axes[j].axis("off")
        axes[j].set_title(f"{j+1}ch", fontsize=20, color='red')

    # fig.suptitle(f"Data No.{index}, min={data.min():.2f}, max={data.max():.2f}", fontsize=30) 
    fig.suptitle(f"Data No.{index}, judge: {judgement_list[index]}", fontsize=30, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{region_name}_channel_map_No{index}_judgement_result.png")
    plt.show()